# Experiments as JSON

An experiment is data: `experiment.json` holds the flow's entry point, the input, the scenarios, the
metrics and the plots, built-in or your own, each named by its `type`. Keep one per folder
(`premium_sweep/experiment.json` here, next to its input) and run, edit or diff it like any other file;
a UI can edit the same file.

In [1]:
from pathlib import Path
from decider.experiments import *

path = Path("example_projects/experiments/premium_sweep/experiment.json")
print(path.read_text())

{
  "version": "2",
  "name": "premium-sweep",
  "description": "Raise the unsecured arm's premium and see how rates, payments and approvals move.",
  "flow": {
    "entry": "example_projects.experiments.loan_pipeline:build",
    "decider": ""
  },
  "revision": "HEAD",
  "input": {
    "path": "applications.parquet"
  },
  "scenarios": [
    {
      "name": "premium_0.06",
      "revision": null,
      "params": {
        "loans": {
          "pricing": {
            "unsecured": {
              "premium": 0.06
            }
          }
        }
      },
      "overrides": {},
      "at": null,
      "row": null
    },
    {
      "name": "premium_0.08",
      "revision": null,
      "params": {
        "loans": {
          "pricing": {
            "unsecured": {
              "premium": 0.08
            }
          }
        }
      },
      "overrides": {},
      "at": null,
      "row": null
    }
  ],
  "metrics": [
    {
      "type": "divergence",
      "name": "",
      "nodes

`Experiment.load` imports the flow from `flow.entry` and scans the input from `input.path`, relative to the
JSON file. The result is an ordinary `Experiment`: run it as is, or add to it first. This one carries its
own `report` layout, so `show()` draws that.

In [2]:
result = Experiment.load(path).run()
result.show()

scenario,status,records,revision,changes,error
baseline,completed,"20,000",,unchanged,
premium_0.06,completed,"20,000",,"params {""loans"": {""pricing"": {""unsecured"": {""premium"": 0.06}}}}",
premium_0.08,completed,"20,000",,"params {""loans"": {""pricing"": {""unsecured"": {""premium"": 0.08}}}}",
scenario,column,compared,changed,first,samples
baseline,approved,"20,000",0,,[]
premium_0.06,approved,"20,000",496,12,"[{""row"": 12, ""expected"": true, ""actual"": false}, {""row"": 16, ""expected"": true..."
premium_0.08,approved,"20,000","4,350",7,"[{""row"": 7, ""expected"": true, ""actual"": false}, {""row"": 9, ""expected"": true, ..."


Editing the JSON is editing the experiment. Here the file is changed as a dict, the way a UI or a script
would, then validated against the flow before anything runs.

In [3]:
import json, tempfile

data = json.loads(path.read_text())
data["scenarios"].append({"name": "no_premium", "params": {"loans": {"pricing": {"unsecured": {"premium": 0.0}}}}})
data["metrics"].append({"type": "describe", "column": "payment"})
data["plots"].append({"type": "ranges", "metric": "describe"})
data["report"]["blocks"].append({"type": "metric_section", "metric": "describe"})
edited = ExperimentDef.model_validate(data)

from example_projects.experiments.loan_pipeline import build
validate(edited, build())

[]

In [4]:
copy = Path(tempfile.mkdtemp()) / "experiment.json"
copy.write_text(edited.model_dump_json(indent=2))
(Path(copy.parent) / "applications.parquet").write_bytes((path.parent / "applications.parquet").read_bytes())

result = Experiment.load(copy).run()
{name: s.changed for name, s in result.summaries["divergence"].items()}

{'baseline': 0, 'premium_0.06': 496, 'premium_0.08': 4350, 'no_premium': 0}

In [5]:
result.plots["describe"]

<vega-lite chart>

A Python-built experiment saves to the same format and loads back equal. For a form-based editor, each
registered metric and plot type has a JSON Schema.

In [6]:
exp = Experiment.load(path)
assert ExperimentDef.model_validate_json(exp.def_.model_dump_json()) == exp.def_
sorted(Metric.json_schema()), sorted(Plot.json_schema())

(['branch_flows', 'describe', 'divergence', 'histogram', 'value_counts'],
 ['bars', 'changes', 'distribution', 'ranges', 'sankey'])